# Ollama API Usage Demonstration

This notebook demonstrates how to use both the native Ollama API and our Python wrapper layer.

## 1. Setup and Imports

In [1]:
import requests
import json
import os
import Ollama_utils as ou
import numpy as np
import matplotlib.pyplot as plt

## 2. Native Ollama API
First, let's demonstrate using the native Ollama API directly.

In [2]:
print("Checking Ollama availability...")
try:
    response = requests.get("http://localhost:11434/api/tags")
    if response.status_code == 200:
        print("✅ Ollama is running")
        models = response.json().get("models", [])
        print(f"Available models: {[model['name'] for model in models]}")
    else:
        print(f"❌ Ollama error: {response.status_code}")
except Exception as e:
    print(f"❌ Ollama connection error: {str(e)}")

Checking Ollama availability...
✅ Ollama is running
Available models: ['llama3:latest']


In [3]:
# Direct API call for demonstration
def generate_search_terms(query, model="llama3"):
    """Use Ollama to generate search terms for a query"""
    url = "http://localhost:11434/api/generate"
    prompt = f"""Given the following search query, extract 3-5 key search terms that would be most effective 
    for finding relevant documents. Return only the terms separated by commas, no explanations.
    
    Query: {query}
    """
    
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    
    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            return response.json().get("response", "")
        else:
            return f"Error: {response.status_code} - {response.text}"
    except Exception as e:
        return f"Error: {str(e)}"

## 3. Python Wrapper Layer
Now let's use our Python wrapper which simplifies these operations.

In [4]:
# Try the native API to generate search terms
query = "How do I implement secure authentication in a web application?"
search_terms = generate_search_terms(query)
print(f"\nFor query: '{query}'")
print(f"Generated search terms: {search_terms}")

# 2. Document Embedding and Indexing
print("\nDemonstrating document embedding and indexing...")

# Create sample documents for demonstration
os.makedirs("demo_docs", exist_ok=True)

sample_docs = [
    ("authentication.md", """
    # Authentication Best Practices
    
    Authentication is critical for web applications. This guide covers:
    
    - Password hashing with bcrypt
    - Multi-factor authentication
    - JWT tokens for API authentication
    - Session management
    - OAuth 2.0 integration
    """),
    
    ("security_overview.md", """
    # Security Overview
    
    A comprehensive security strategy includes:
    
    - Authentication and authorization
    - Input validation and sanitization
    - HTTPS/TLS encryption
    - Regular security audits
    - Data encryption at rest
    """),
    
    ("api_design.md", """
    # API Design Guide
    
    Building robust APIs requires consideration of:
    
    - Authentication mechanisms (JWT, OAuth)
    - Rate limiting
    - Versioning strategy
    - Error handling
    - Documentation
    """)
]


For query: 'How do I implement secure authentication in a web application?'
Generated search terms: Security, Authentication, Web Application

Demonstrating document embedding and indexing...


In [5]:
# Create sample files
for filename, content in sample_docs:
    with open(f"demo_docs/{filename}", "w") as f:
        f.write(content)
print(f"Created {len(sample_docs)} sample documents in 'demo_docs/'")

Created 3 sample documents in 'demo_docs/'


In [6]:
# Get embedding model
model = ou.get_embedding_model()
print(f"Using embedding model: {model.__class__.__name__}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

Using embedding model: SentenceTransformer
Embedding dimension: 768


In [7]:
# Index the documents
print("\nIndexing documents...")
file_paths = ou.scan_directory("demo_docs")
print(f"Found {len(file_paths)} files")


Indexing documents...
Found 3 files


In [8]:
# Track progress
def progress_callback(progress, message):
    print(f"Progress: {progress*100:.1f}% - {message}")

success = ou.build_document_index(
    file_paths,
    index_path="demo_docs/index.bin",
    metadata_path="demo_docs/metadata.pkl",
    progress_callback=progress_callback
)
print(f"Indexing {'completed successfully' if success else 'failed'}")

Building index for 3 files
Processing 3 new files
Using 8 parallel workers
Error saving to cache for demo_docs\api_design.md: [Errno 2] No such file or directory: 'index/cache\\c__Users_proto_Local_LLM_tutorials_DATA605_Spring2025_projects_TutorTask125_Spring2025_Local_AI-Powered_Document_Search_Engine_with_Ollama_demo_docs_api_design.md_1747449646.6873598_224.pkl'
Progress: 30.0% - Processed 1/3 files
Error saving to cache for demo_docs\security_overview.md: [Errno 2] No such file or directory: 'index/cache\\c__Users_proto_Local_LLM_tutorials_DATA605_Spring2025_projects_TutorTask125_Spring2025_Local_AI-Powered_Document_Search_Engine_with_Ollama_demo_docs_security_overview.md_1747449646.6863585_255.pkl'
Progress: 60.0% - Processed 2/3 files
Error saving to cache for demo_docs\authentication.md: [Errno 2] No such file or directory: 'index/cache\\c__Users_proto_Local_LLM_tutorials_DATA605_Spring2025_projects_TutorTask125_Spring2025_Local_AI-Powered_Document_Search_Engine_with_Ollama_demo

In [9]:
# 3. Document Search
print("\nSearching for documents...")
search_queries = [
    "authentication methods for web apps",
    "API security best practices",
    "how to protect user data"
]


Searching for documents...


In [10]:
# Test search functionality
for query in search_queries:
    print(f"\nSearching for: '{query}'")
    results = ou.search_documents(
        query,
        top_k=2,
        index_path="demo_docs/index.bin",
        metadata_path="demo_docs/metadata.pkl"
    )
    
    if "error" in results:
        print(f"Error: {results['error']}")
    else:
        print(f"Found {len(results)} results:")
        for i, result in enumerate(results):
            print(f"\n[{i+1}] {result['filename']} (Score: {result['score']:.3f})")
            print(f"Snippet: {result['snippet'][:150]}...")


Searching for: 'authentication methods for web apps'
Found 2 results:

[1] authentication.md (Score: 0.675)
Snippet: 
    # Authentication Best Practices

    Authentication is critical for web applications. This guide covers:

    - Password hashing with bcrypt
    ...

[2] api_design.md (Score: 0.466)
Snippet: 
    # API Design Guide

    Building robust APIs requires consideration of:

    - Authentication mechanisms (JWT, OAuth)
    - Rate limiting
    - V...

Searching for: 'API security best practices'
Found 2 results:

[1] api_design.md (Score: 0.684)
Snippet: 
    # API Design Guide

    Building robust APIs requires consideration of:

    - Authentication mechanisms (JWT, OAuth)
    - Rate limiting
    - V...

[2] authentication.md (Score: 0.540)
Snippet: 
    # Authentication Best Practices

    Authentication is critical for web applications. This guide covers:

    - Password hashing with bcrypt
    ...

Searching for: 'how to protect user data'
Found 2 results:

[1] secur

In [11]:
# 4. Search Enhancement with Ollama
print("\nEnhancing search with Ollama...")
query = "secure login implementation"

# First get raw search results
print(f"Original query: '{query}'")
raw_results = ou.search_documents(
    query,
    top_k=2,
    index_path="demo_docs/index.bin",
    metadata_path="demo_docs/metadata.pkl"
)

# Use Ollama to enhance the query
enhanced_query = ou.query_ollama(
    f"Rewrite this search query to be more comprehensive for finding technical documentation: '{query}'. Return only the enhanced query and nothing else in beginning or end.",
    model="llama3"
)
print(f"Enhanced query: '{enhanced_query}'")


Enhancing search with Ollama...
Original query: 'secure login implementation'
Enhanced query: '"secure login implementation OR secure authentication process OR secure access control mechanisms OR secure single sign-on solutions OR secure user registration procedures OR secure password management techniques AND (technical documentation OR developer guide OR API reference OR software manual)"'


In [12]:
# Search with enhanced query
enhanced_results = ou.search_documents(
    enhanced_query,
    top_k=2,
    index_path="demo_docs/index.bin",
    metadata_path="demo_docs/metadata.pkl"
)

# Compare results
print("\nComparison of search results:")
print("Original query results:")
for i, result in enumerate(raw_results):
    print(f"[{i+1}] {result['filename']} (Score: {result['score']:.3f})")

print("\nEnhanced query results:")
for i, result in enumerate(enhanced_results):
    print(f"[{i+1}] {result['filename']} (Score: {result['score']:.3f})")

# 5. Cleanup
print("\nCleaning up demo files...")
# Uncomment to remove demo files
import shutil
shutil.rmtree("demo_docs")
print("Demo complete!")


Comparison of search results:
Original query results:
[1] authentication.md (Score: 0.546)
[2] security_overview.md (Score: 0.483)

Enhanced query results:
[1] security_overview.md (Score: 0.665)
[2] authentication.md (Score: 0.540)

Cleaning up demo files...
Demo complete!
